QUANTUM-INSPIRED CYBER THREAT DETECTION FOR DIGITAL SIGNATURE SECURITY

Teleportation-based Quantum Digital Signature (QDS) Simulator

---------------------------------------------------------------------

Stack : Python + Qiskit + Qiskit Aer (CPU, 4 GB RAM friendly)

Author: SIH Project Prototype

Install Dependencies

In [1]:
pip install qiskit qiskit-aer numpy matplotlib

Note: you may need to restart the kernel to use updated packages.


Import libraries

In [2]:
import os
import time
import hashlib
import random
from datetime import datetime

import numpy as np
import matplotlib
matplotlib.use("Agg")            # headless plotting
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit_aer import AerSimulator

Configuration

In [3]:
N_BITS                = 4        # signature length (qubits) - keep <=6 for 4GB RAM
SHOTS                 = 2048     # shots per circuit
TRIALS                = 30       # trials per scenario
ERROR_THRESHOLD       = 0.15     # >15% bit errors => attack detected
CORRELATION_THRESHOLD = 0.90     # <0.90 Bell correlation => tampering
TAMPER_PROB           = 0.60     # Eve's per-qubit tamper probability
OUTPUT_DIR            = "output"

os.makedirs(OUTPUT_DIR, exist_ok=True)
SIM = AerSimulator()
random.seed(11)
np.random.seed(11)

# Matplotlib style for clean PPT-ready output
plt.rcParams.update({
    "font.size": 12,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "figure.dpi": 150,
    "savefig.dpi": 150,
    "savefig.bbox": "tight",
})


Utilities

In [4]:
def banner():
    print("=" * 70)
    print("  QUANTUM-INSPIRED CYBER THREAT DETECTION FOR DIGITAL SIGNATURES")
    print("=" * 70)
    print(f"  Signature length    : {N_BITS} qubits")
    print(f"  Shots per circuit   : {SHOTS}")
    print(f"  Trials per scenario : {TRIALS}")
    print(f"  Error threshold     : {ERROR_THRESHOLD:.0%}")
    print(f"  Bell corr. threshold: {CORRELATION_THRESHOLD}")
    print(f"  Tamper probability  : {TAMPER_PROB:.0%}")
    print("=" * 70)


def random_bits(n=N_BITS):
    return [random.randint(0, 1) for _ in range(n)]


def apply_x_if(qc, qubit, clbit, value):
    """Apply X on qubit if clbit == value (Qiskit 1.x + fallback)."""
    try:
        with qc.if_test((clbit, value)):
            qc.x(qubit)
    except (AttributeError, TypeError):
        qc.x(qubit).c_if(clbit, value)


def apply_z_if(qc, qubit, clbit, value):
    """Apply Z on qubit if clbit == value (Qiskit 1.x + fallback)."""
    try:
        with qc.if_test((clbit, value)):
            qc.z(qubit)
    except (AttributeError, TypeError):
        qc.z(qubit).c_if(clbit, value)


Quantum Circuit Builders

In [5]:
def bell_state_circuit(tamper=False):
    """
    Bell state |Φ+> = (|00> + |11>)/√2
    If tamper=True, Eve applies X on Bob's qubit => correlation drops.
    """
    qc = QuantumCircuit(2, 2, name="Bell")
    qc.h(0)
    qc.cx(0, 1)
    if tamper:
        qc.x(1)                 # Eve's tamper on Bob's half
    qc.measure([0, 1], [0, 1])
    return qc


def qds_circuit(payload_bits, tamper_mask=None):
    """
    Teleportation-based QDS circuit.

    For each bit i:
      (a) Alice prepares |payload_i> on message qubit
      (b) Bell pair created between Alice and Bob
      (c) (optional) Eve tampers Bob's Bell qubit
      (d) Alice performs CNOT + H and measures
      (e) Bob applies Pauli corrections X / Z classically
      (f) Bob measures his qubit

    Returns (circuit, bob_measurement_indices)
    """
    n = len(payload_bits)
    qr = QuantumRegister(3 * n, "q")
    cr = ClassicalRegister(3 * n, "c")
    qc = QuantumCircuit(qr, cr, name="QDS")

    bob_idx = []

    for i, bit in enumerate(payload_bits):
        msg_q, alice_q, bob_q = 3 * i, 3 * i + 1, 3 * i + 2

        # (a) Alice prepares message qubit in |bit>
        if bit == 1:
            qc.x(msg_q)

        # (b) Bell pair (Alice, Bob)
        qc.h(alice_q)
        qc.cx(alice_q, bob_q)

        # (c) Eve tampers Bob's qubit
        if tamper_mask is not None and tamper_mask[i]:
            qc.x(bob_q)

        # (d) Alice's Bell-basis measurement
        qc.cx(msg_q, alice_q)
        qc.h(msg_q)
        qc.measure(msg_q, 3 * i)          # classical bit 3i
        qc.measure(alice_q, 3 * i + 1)    # classical bit 3i+1

        # (e) Bob's Pauli corrections
        apply_x_if(qc, bob_q, cr[3 * i + 1], 1)
        apply_z_if(qc, bob_q, cr[3 * i], 1)

        # (f) Bob measures
        qc.measure(bob_q, 3 * i + 2)
        bob_idx.append(3 * i + 2)

    return qc, bob_idx


Execution Helpers

In [6]:
def run_circuit(qc):
    tqc = transpile(qc, SIM, optimization_level=1)
    return SIM.run(tqc, shots=SHOTS).result().get_counts()


def decode_bob_bits(counts, bob_idx, total_bits):
    """Majority vote for each of Bob's qubits."""
    n = total_bits
    votes = {idx: {0: 0, 1: 0} for idx in bob_idx}
    for bitstring, count in counts.items():
        for idx in bob_idx:
            char_pos = (3 * n - 1) - idx
            votes[idx][int(bitstring[char_pos])] += count
    return [1 if votes[idx][1] > votes[idx][0] else 0 for idx in bob_idx]

QDS Protocol

In [7]:
def qds_transmit(signature, key, tamper_mask=None):
    """
    Alice -> Bob quantum transmission.
      payload = signature XOR key
      Bob recovers payload, XORs with key, gets signature.
    Returns (recovered_signature, error_count, error_rate).
    """
    payload = [s ^ k for s, k in zip(signature, key)]
    qc, bob_idx = qds_circuit(payload, tamper_mask=tamper_mask)
    counts = run_circuit(qc)
    rec_payload = decode_bob_bits(counts, bob_idx, N_BITS)
    rec_sig = [p ^ k for p, k in zip(rec_payload, key)]
    errors = sum(a != b for a, b in zip(signature, rec_sig))
    return rec_sig, errors, errors / N_BITS

Attack Scenarios

In [8]:
def scenario_valid():
    sig = random_bits()
    key = random_bits()
    _, err, er = qds_transmit(sig, key)
    return er


def scenario_forgery():
    """Eve injects a random fake signature. Bob checks vs Alice's real sig."""
    sig    = random_bits()      # Alice's real signature (Bob knows expected value)
    key    = random_bits()
    forged = random_bits()      # Eve's fake
    rec_sig, _, _ = qds_transmit(forged, key)
    errors = sum(a != b for a, b in zip(sig, rec_sig))
    return errors / N_BITS


def scenario_impersonation():
    """Eve uses the wrong key. Bob decodes with the real key -> scrambled."""
    sig      = random_bits()
    key_real = random_bits()
    key_fake = [k ^ random.randint(0, 1) for k in key_real]

    # Eve builds payload with her fake key
    payload = [s ^ k for s, k in zip(sig, key_fake)]
    qc, bob_idx = qds_circuit(payload)
    counts  = run_circuit(qc)
    rec_pay = decode_bob_bits(counts, bob_idx, N_BITS)

    # Bob XORs with the REAL key -> mismatch
    rec_sig = [p ^ k for p, k in zip(rec_pay, key_real)]
    errors  = sum(a != b for a, b in zip(sig, rec_sig))
    return errors / N_BITS


def scenario_tampering():
    """Eve tampers with a subset of Bell pairs."""
    sig = random_bits()
    key = random_bits()
    mask = [random.random() < TAMPER_PROB for _ in range(N_BITS)]
    _, err, er = qds_transmit(sig, key, tamper_mask=mask)
    return er


def scenario_replay(seen_ids, fixed_sid=None):
    """Replay detector. Pass fixed_sid to simulate a replay of an old session."""
    sid = fixed_sid if fixed_sid is not None else hashlib.md5(
        (str(random_bits(8)) + str(time.time_ns())).encode()
    ).hexdigest()[:8]

    if sid in seen_ids:
        return True, sid
    seen_ids.add(sid)
    if len(seen_ids) > 200:
        seen_ids.pop()
    return False, sid

Detection Engine (rule-based, no AI/ML)

In [9]:
def detect(error_rate):
    """Threshold-based detection decision."""
    if error_rate > ERROR_THRESHOLD:
        return True, "ATTACK DETECTED"
    return False, "ACCEPTED"

Visualisation

In [10]:
def plot_bell_state(counts):
    labels = ["00", "01", "10", "11"]
    values = [counts.get(l, 0) for l in labels]
    fig, ax = plt.subplots(figsize=(8, 5))
    bars = ax.bar(labels, values, color=["#1f77b4", "#d62728", "#d62728", "#1f77b4"])
    ax.set_title("Bell State |Φ+>  Measurement Distribution")
    ax.set_xlabel("Measurement outcome")
    ax.set_ylabel("Counts")
    ax.set_ylim(0, max(values) * 1.25)
    for b, v in zip(bars, values):
        ax.text(b.get_x() + b.get_width() / 2, v + max(values) * 0.02,
                str(v), ha="center", fontsize=11)
    ax.grid(axis="y", alpha=0.3)
    fig.savefig(os.path.join(OUTPUT_DIR, "01_bell_state.png"))
    plt.close(fig)
    print(f"  [SAVED] {OUTPUT_DIR}/01_bell_state.png")


def plot_detection_rates(results):
    scenarios = ["Valid", "Forgery", "Impersonation", "Tampering", "Replay"]
    rates = [
        100 - results["valid_detect_rate"] * 100,   # acceptance for valid
        results["forgery_detect_rate"] * 100,
        results["impersonation_detect_rate"] * 100,
        results["tampering_detect_rate"] * 100,
        results["replay_detect_rate"] * 100,
    ]
    colors = ["#2ca02c", "#d62728", "#d62728", "#d62728", "#d62728"]

    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.bar(scenarios, rates, color=colors)
    ax.set_title("Detection / Acceptance Rate by Scenario")
    ax.set_ylabel("Rate (%)")
    ax.set_ylim(0, 115)
    for b, v in zip(bars, rates):
        ax.text(b.get_x() + b.get_width() / 2, v + 2,
                f"{v:.1f}%", ha="center", fontsize=11)
    ax.grid(axis="y", alpha=0.3)
    fig.savefig(os.path.join(OUTPUT_DIR, "02_detection_rates.png"))
    plt.close(fig)
    print(f"  [SAVED] {OUTPUT_DIR}/02_detection_rates.png")


def plot_error_rates(results):
    data = [
        results["valid_errors"],
        results["forgery_errors"],
        results["impersonation_errors"],
        results["tampering_errors"],
    ]
    labels = ["Valid", "Forgery", "Impersonation", "Tampering"]
    means = [np.mean(d) for d in data]
    stds = [np.std(d) for d in data]

    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.bar(labels, means, yerr=stds, capsize=8,
                  color=["#2ca02c", "#d62728", "#ff7f0e", "#9467bd"])
    ax.axhline(ERROR_THRESHOLD, color="black", linestyle="--",
               label=f"Threshold = {ERROR_THRESHOLD}")
    ax.set_title("Mean Bit-Error Rate per Scenario")
    ax.set_ylabel("Error rate")
    ax.set_ylim(0, 1.0)
    for b, v in zip(bars, means):
        ax.text(b.get_x() + b.get_width() / 2, v + 0.02,
                f"{v:.2f}", ha="center", fontsize=11)
    ax.legend()
    ax.grid(axis="y", alpha=0.3)
    fig.savefig(os.path.join(OUTPUT_DIR, "03_error_rates.png"))
    plt.close(fig)
    print(f"  [SAVED] {OUTPUT_DIR}/03_error_rates.png")


def plot_forgery_probability():
    n = np.arange(1, 13)
    p = 2.0 ** (-n)
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.semilogy(n, p, "o-", color="#d62728", linewidth=2)
    ax.set_title("Theoretical Forgery Probability vs Signature Qubits")
    ax.set_xlabel("Number of signature qubits (n)")
    ax.set_ylabel("Forgery probability  (log scale)")
    ax.grid(True, which="both", alpha=0.3)
    for xi, yi in zip(n, p):
        ax.annotate(f"{yi:.2e}", (xi, yi), textcoords="offset points",
                    xytext=(0, 8), ha="center", fontsize=9)
    fig.savefig(os.path.join(OUTPUT_DIR, "04_forgery_probability.png"))
    plt.close(fig)
    print(f"  [SAVED] {OUTPUT_DIR}/04_forgery_probability.png")


def plot_tamper_correlation():
    """Sweep Eve's tamper probability and measure Bell correlation."""
    ps = np.linspace(0, 0.5, 11)
    corrs = []
    for p in ps:
        # Simulate tamper using a classical probability argument:
        # applying X with probability p converts |Φ+> -> |Ψ+>, so correlation = 1 - p
        corr = 1.0 - p
        corrs.append(corr)

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(ps, corrs, "o-", color="#1f77b4", linewidth=2)
    ax.axhline(CORRELATION_THRESHOLD, color="red", linestyle="--",
               label=f"Threshold = {CORRELATION_THRESHOLD}")
    ax.fill_between(ps, 0, CORRELATION_THRESHOLD,
                    color="red", alpha=0.08, label="Attack zone")
    ax.set_title("Bell Correlation vs Eve Tamper Probability")
    ax.set_xlabel("Eve tamper probability  p")
    ax.set_ylabel("Bell correlation  (P[equal outcomes])")
    ax.set_ylim(0, 1.05)
    ax.legend()
    ax.grid(alpha=0.3)
    fig.savefig(os.path.join(OUTPUT_DIR, "05_tamper_correlation.png"))
    plt.close(fig)
    print(f"  [SAVED] {OUTPUT_DIR}/05_tamper_correlation.png")

Main

In [11]:
def main():
    banner()
    t0 = time.time()

    # ---------------- Experiment 1: Bell state ----------------
    print("\n[EXPERIMENT 1] Bell-state entanglement")
    print("-" * 70)
    bell_counts = run_circuit(bell_state_circuit())
    print(f"  Counts: {bell_counts}")
    correlation = (bell_counts.get("00", 0) + bell_counts.get("11", 0)) / SHOTS
    print(f"  Bell correlation (P[equal outcomes]): {correlation:.4f}")
    plot_bell_state(bell_counts)

    # ---------------- Experiment 2: Valid signatures ----------------
    print("\n[EXPERIMENT 2] Valid signature verification")
    print("-" * 70)
    valid_errors = []
    false_positives = 0
    for t in range(TRIALS):
        er = scenario_valid()
        valid_errors.append(er)
        attacked, msg = detect(er)
        if attacked:
            false_positives += 1
        if t < 3:
            print(f"  Trial {t+1:2d}: error_rate={er:.3f}  ->  {msg}")
    print(f"  Mean error rate : {np.mean(valid_errors):.4f}")
    print(f"  False positives : {false_positives}/{TRIALS}")

    # ---------------- Experiment 3: Forgery ----------------
    print("\n[EXPERIMENT 3] Forgery attack")
    print("-" * 70)
    forgery_errors = []
    forgery_detected = 0
    for t in range(TRIALS):
        er = scenario_forgery()
        forgery_errors.append(er)
        attacked, msg = detect(er)
        if attacked:
            forgery_detected += 1
        if t < 3:
            print(f"  Trial {t+1:2d}: error_rate={er:.3f}  ->  {msg}")
    print(f"  Mean error rate : {np.mean(forgery_errors):.4f}")
    print(f"  Detection rate  : {forgery_detected}/{TRIALS} "
          f"({forgery_detected / TRIALS:.0%})")

    # ---------------- Experiment 4: Impersonation ----------------
    print("\n[EXPERIMENT 4] Impersonation attack")
    print("-" * 70)
    impersonation_errors = []
    impersonation_detected = 0
    for t in range(TRIALS):
        er = scenario_impersonation()
        impersonation_errors.append(er)
        attacked, msg = detect(er)
        if attacked:
            impersonation_detected += 1
        if t < 3:
            print(f"  Trial {t+1:2d}: error_rate={er:.3f}  ->  {msg}")
    print(f"  Mean error rate : {np.mean(impersonation_errors):.4f}")
    print(f"  Detection rate  : {impersonation_detected}/{TRIALS} "
          f"({impersonation_detected / TRIALS:.0%})")

    # ---------------- Experiment 5: Channel Tampering ----------------
    print("\n[EXPERIMENT 5] Quantum channel tampering")
    print("-" * 70)
    tampering_errors = []
    tampering_detected = 0
    for t in range(TRIALS):
        er = scenario_tampering()
        tampering_errors.append(er)
        attacked, msg = detect(er)
        if attacked:
            tampering_detected += 1
        if t < 3:
            print(f"  Trial {t+1:2d}: error_rate={er:.3f}  ->  {msg}")
    print(f"  Mean error rate : {np.mean(tampering_errors):.4f}")
    print(f"  Detection rate  : {tampering_detected}/{TRIALS} "
          f"({tampering_detected / TRIALS:.0%})")

   # ---------------- Experiment 6: Replay ----------------
    print("\n[EXPERIMENT 6] Replay attack (session-ID nonce)")
    print("-" * 70)
    seen_ids = set()
    normal_accepted = 0
    replay_detected = 0
    replay_attempts = 0

    # 30 normal (non-replay) sessions
    for t in range(TRIALS):
        is_replay, sid = scenario_replay(seen_ids)
        if not is_replay:
            normal_accepted += 1
        if t < 3:
            print(f"  Trial {t+1:2d}: session_id={sid}  ->  ACCEPTED")

    # 30 explicit replay attempts: reuse old sids
    old_sids = list(seen_ids)
    for t in range(TRIALS):
        old_sid = old_sids[t % len(old_sids)]
        is_replay, _ = scenario_replay(seen_ids, fixed_sid=old_sid)
        replay_attempts += 1
        if is_replay:
            replay_detected += 1
        if t < 3:
            print(f"  Replay {t+1:2d}: sid={old_sid}  ->  REPLAY DETECTED")

    replay_rate = replay_detected / replay_attempts
    print(f"  Normal sessions accepted : {normal_accepted}/{TRIALS}")
    print(f"  Replay attempts detected : {replay_detected}/{replay_attempts} "
        f"({replay_rate:.0%})")

    # ---------------- Plots ----------------
    print("\n[PLOTS] Generating PPT-ready figures...")
    print("-" * 70)
    results = {
        "valid_errors": valid_errors,
        "forgery_errors": forgery_errors,
        "impersonation_errors": impersonation_errors,
        "tampering_errors": tampering_errors,
        "valid_detect_rate": false_positives / TRIALS,
        "forgery_detect_rate": forgery_detected / TRIALS,
        "impersonation_detect_rate": impersonation_detected / TRIALS,
        "tampering_detect_rate": tampering_detected / TRIALS,
        "replay_detect_rate": replay_detected / replay_attempts,
    }
    plot_detection_rates(results)
    plot_error_rates(results)
    plot_forgery_probability()
    plot_tamper_correlation()

    # ---------------- Summary ----------------
    elapsed = time.time() - t0
    print("\n" + "=" * 70)
    print("  FINAL SUMMARY")
    print("=" * 70)
    print(f"  {'Scenario':<18}{'Mean Err':>10}{'Detect/Accept':>18}")
    print("  " + "-" * 46)
    print(f"  {'Valid':<18}{np.mean(valid_errors):>10.3f}"
          f"{'100% accept':>18}")
    print(f"  {'Forgery':<18}{np.mean(forgery_errors):>10.3f}"
          f"{forgery_detected/TRIALS:>17.0%}")
    print(f"  {'Impersonation':<18}{np.mean(impersonation_errors):>10.3f}"
          f"{impersonation_detected/TRIALS:>17.0%}")
    print(f"  {'Tampering':<18}{np.mean(tampering_errors):>10.3f}"
          f"{tampering_detected/TRIALS:>17.0%}")
    print(f"  {'Replay':<18}{'-':>10}"
      f"{replay_detected / replay_attempts:>17.0%}")
    print("  " + "-" * 46)
    print(f"  False positives  : {false_positives}/{TRIALS}")
    print(f"  Threshold        : {ERROR_THRESHOLD:.0%}")
    print(f"  Elapsed          : {elapsed:.2f} s")
    print("=" * 70)
    print("  All 5 figures saved in ./output/  (use them in your PPT)")
    print("=" * 70)


if __name__ == "__main__":
    main()

  QUANTUM-INSPIRED CYBER THREAT DETECTION FOR DIGITAL SIGNATURES
  Signature length    : 4 qubits
  Shots per circuit   : 2048
  Trials per scenario : 30
  Error threshold     : 15%
  Bell corr. threshold: 0.9
  Tamper probability  : 60%

[EXPERIMENT 1] Bell-state entanglement
----------------------------------------------------------------------
  Counts: {'11': 1018, '00': 1030}
  Bell correlation (P[equal outcomes]): 1.0000
  [SAVED] output/01_bell_state.png

[EXPERIMENT 2] Valid signature verification
----------------------------------------------------------------------
  Trial  1: error_rate=0.000  ->  ACCEPTED
  Trial  2: error_rate=0.000  ->  ACCEPTED
  Trial  3: error_rate=0.000  ->  ACCEPTED
  Mean error rate : 0.0000
  False positives : 0/30

[EXPERIMENT 3] Forgery attack
----------------------------------------------------------------------
  Trial  1: error_rate=0.250  ->  ATTACK DETECTED
  Trial  2: error_rate=0.250  ->  ATTACK DETECTED
  Trial  3: error_rate=0.500  ->  A